# LangGraph State with Reducers: Practice Exercise

Build the state class for an order processing pipeline using Pydantic with reducers. You will define state fields with appropriate reducers and return correct state updates from nodes.

**What you'll implement:**
- Define a State class with fields that use `operator.add` reducer for accumulation
- Define fields that use default (overwrite) behavior
- Return correct partial state updates from node functions

**Estimated time:** 10 minutes

## Setup

Run this cell to import all required libraries.

In [1]:
# Setup - run this cell first

from typing import Annotated,List
from operator import add
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

print("Setup complete!")

Setup complete!


## Context

You are building an order processing pipeline that tracks an order through multiple stages. The pipeline has three nodes:

1. **Validate Order**: Checks the order and adds validation messages
2. **Calculate Totals**: Computes pricing and adds calculation messages
3. **Finalize Order**: Prepares the order for shipping and adds final messages

**State Requirements:**
- `order_id`: The order identifier (e.g., "ORD-12345") - stays constant throughout
- `items`: List of items being ordered (e.g., `["Widget", "Gadget"]`) - stays constant
- `log`: A list of processing messages that **accumulates** from each node
- `stage`: Current processing stage that gets **overwritten** by each node

**Expected Output:** After running the graph with `order_id="ORD-001"` and `items=["Laptop", "Mouse"]`:
- `log` should contain messages from ALL three nodes (accumulated)
- `stage` should be `"complete"` (the final stage)

## Step 1: Define the State Class

Create a State class using Pydantic's `BaseModel`. Use `Annotated[list[str], add]` for fields that should accumulate values, and regular types for fields that should be overwritten.

In [2]:
class OrderState(BaseModel):
    """
    State for order processing pipeline.

    Fields:
        order_id: The order identifier string (stays constant, no reducer needed)
        items: List of item names being ordered (stays constant, no reducer needed)
        log: Processing messages that ACCUMULATE from each node (use add reducer)
        stage: Current processing stage that gets OVERWRITTEN (default behavior)

    Default values:
        - order_id: empty string
        - items: empty list (use Field with default_factory)
        - log: empty list (use Field with default_factory)
        - stage: "pending"
    """
    # TODO: Define order_id as a string with default empty string
    order_id:str = ""
    # TODO: Define items as a list of strings with default_factory=list
    items:List[str] = Field(default_factory=list)
    # TODO: Define log with Annotated[list[str], add] reducer and default_factory=list
    log:Annotated[list[str], add] = Field(default_factory=list)
    # TODO: Define stage as a string with default "pending"
    stage:str = "pending"

## Step 2: Complete the Node Return Statements

Each node function is provided with logic to generate messages. You need to complete the return statements to update state correctly:
- Return `log` as a **list** (it will be appended to existing log entries)
- Return `stage` as a **string** (it will overwrite the previous stage)

In [3]:
def validate_order(state: OrderState) -> dict:
    """
    Validate the order and log validation results.

    This node checks if the order has items and creates validation messages.

    Args:
        state: Current state containing order_id and items

    Returns:
        Dictionary with:
        - 'log': list of validation messages (will be APPENDED)
        - 'stage': set to "validated" (will OVERWRITE)
    """
    messages = []
    messages.append(f"Validating order {state.order_id}")

    if state.items:
        messages.append(f"Order contains {len(state.items)} item(s): {', '.join(state.items)}")
    else:
        messages.append("Warning: Order has no items")

    # TODO: Return a dict with 'log' (the messages list) and 'stage' (set to "validated")
    return {"log": messages, "stage": "validated"}

In [4]:
def calculate_totals(state: OrderState) -> dict:
    """
    Calculate order totals and log the calculations.

    This node computes a mock total based on number of items.

    Args:
        state: Current state containing items list

    Returns:
        Dictionary with:
        - 'log': list of calculation messages (will be APPENDED)
        - 'stage': set to "calculated" (will OVERWRITE)
    """
    item_count = len(state.items)
    subtotal = item_count * 99.99
    tax = subtotal * 0.08
    total = subtotal + tax

    messages = [
        f"Subtotal: ${subtotal:.2f}",
        f"Tax (8%): ${tax:.2f}",
        f"Total: ${total:.2f}"
    ]

    # TODO: Return a dict with 'log' (the messages list) and 'stage' (set to "calculated")
    return {"log": messages, "stage": "calculated"}

In [5]:
def finalize_order(state: OrderState) -> dict:
    """
    Finalize the order for shipping and log completion.

    This node marks the order as ready for fulfillment.

    Args:
        state: Current state containing order_id

    Returns:
        Dictionary with:
        - 'log': list of finalization messages (will be APPENDED)
        - 'stage': set to "complete" (will OVERWRITE)
    """
    messages = [
        f"Order {state.order_id} finalized",
        "Ready for fulfillment"
    ]

    # TODO: Return a dict with 'log' (the messages list) and 'stage' (set to "complete")
    return {"log": messages, "stage": "complete" }

## Graph Structure (Provided)

The graph is built for you. This cell will work once you complete the state class and node return statements.

In [6]:
# Build the graph - this is provided for you
builder = StateGraph(OrderState)

builder.add_node("validate", validate_order)
builder.add_node("calculate", calculate_totals)
builder.add_node("finalize", finalize_order)

builder.add_edge(START, "validate")
builder.add_edge("validate", "calculate")
builder.add_edge("calculate", "finalize")
builder.add_edge("finalize", END)

graph = builder.compile()
print("Graph compiled successfully!")

Graph compiled successfully!


## Run Your Implementation

Test your state class and node implementations.

In [7]:
# Run the order processing pipeline
result = graph.invoke({
    "order_id": "ORD-001",
    "items": ["Laptop", "Mouse"]
})

print("=" * 50)
print("ORDER PROCESSING RESULTS")
print("=" * 50)
print(f"\nOrder ID: {result['order_id']}")
print(f"Items: {result['items']}")
print(f"Final Stage: {result['stage']}")
print(f"\nProcessing Log (should have entries from ALL nodes):")
for i, entry in enumerate(result['log'], 1):
    print(f"  {i}. {entry}")

ORDER PROCESSING RESULTS

Order ID: ORD-001
Items: ['Laptop', 'Mouse']
Final Stage: complete

Processing Log (should have entries from ALL nodes):
  1. Validating order ORD-001
  2. Order contains 2 item(s): Laptop, Mouse
  3. Subtotal: $199.98
  4. Tax (8%): $16.00
  5. Total: $215.98
  6. Order ORD-001 finalized
  7. Ready for fulfillment


In [8]:
# Verify your implementation
print("\nVerification:")
print(f"  - Log has {len(result['log'])} entries (expected: 7)")
print(f"  - Final stage is '{result['stage']}' (expected: 'complete')")

if len(result['log']) == 7 and result['stage'] == 'complete':
    print("\nSuccess! Your state management is working correctly.")
else:
    print("\nCheck your implementation - log should accumulate and stage should overwrite.")


Verification:
  - Log has 7 entries (expected: 7)
  - Final stage is 'complete' (expected: 'complete')

Success! Your state management is working correctly.
